Building an emotion detection model using LSTM

Let's get the dataset first

In [3]:
import pandas as pd
df=pd.read_csv("emotion_dataset (1).csv")
df.head()

,sentence,emotion
0,How dare they lie to my face like that!,anger
1,I used to look forward to mornings but now I j...,sadness
2,There was mold all over the leftovers in the f...,disgust
3,I was trembling with rage by the time the meet...,anger
4,She updated the spreadsheet and shared it with...,neutral


In [4]:
X=df.drop(["emotion"],axis=1)
X.head()

,sentence
0,How dare they lie to my face like that!
1,I used to look forward to mornings but now I j...
2,There was mold all over the leftovers in the f...
3,I was trembling with rage by the time the meet...
4,She updated the spreadsheet and shared it with...


In [11]:
y=df["emotion"]
y.head()

,emotion
0,anger
1,sadness
2,disgust
3,anger
4,neutral


Let's start preprocessing the dataset

Encode labels (y)

In [12]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
label=encoder.fit_transform(y)
print(label[:6])
y.head()

[0 5 1 0 4 5]


,emotion
0,anger
1,sadness
2,disgust
3,anger
4,neutral


Tokenization to make sequential inputs

In [18]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X['sentence'])
sequences=tokenizer.texts_to_sequences(X['sentence'])
print("Sample sequences:")
print(sequences[:5])
print("\nWord index (first 10 items):")
print(dict(list(tokenizer.word_index.items())[:10]))

Sample sequences:
[[92, 139, 13, 256, 7, 3, 140, 26, 28], [2, 141, 7, 70, 257, 7, 258, 52, 142, 2, 27, 143, 144], [53, 5, 145, 40, 36, 1, 259, 9, 1, 260], [2, 5, 261, 16, 262, 54, 1, 37, 1, 93, 263], [11, 264, 1, 265, 4, 266, 21, 16, 1, 71]]

Word index (first 10 items):
{'the': 1, 'i': 2, 'my': 3, 'and': 4, 'was': 5, 'a': 6, 'to': 7, 'me': 8, 'in': 9, 'of': 10}


Making all sequences to same length

In [21]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X=pad_sequences(sequences,maxlen=25)

In [22]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,label,test_size=0.2,random_state=42)

Building the model

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense
model=Sequential(
    [
        Embedding(input_dim=5000,output_dim=64),
        LSTM(64),
        Dense(32,activation="relu"),
        Dense(len(y),activation="softmax")
    ]
)

Compiling the model

In [26]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

Training the model

In [27]:
model.fit(
    X_train,y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test,y_test)
)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 119ms/step - accuracy: 0.0530 - loss: 5.2341 - val_accuracy: 0.1579 - val_loss: 5.2172
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.1921 - loss: 5.1980 - val_accuracy: 0.2105 - val_loss: 5.1474
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.1921 - loss: 5.0625 - val_accuracy: 0.1053 - val_loss: 4.8276
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.1722 - loss: 4.6240 - val_accuracy: 0.0526 - val_loss: 4.2842
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.1656 - loss: 4.0746 - val_accuracy: 0.0526 - val_loss: 3.7309
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.1921 - loss: 3.5181 - val_accuracy: 0.2105 - val_loss: 3.1297
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.1258 - loss: 2.9509 - val_accuracy: 0.2105 - val_loss: 2.5967
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.1258 - loss: 2.4866 - val_accuracy: 0.2105 - val_loss: 2.2834

In [29]:
def predict_emotion(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=20)

    pred = model.predict(padded)
    label = encoder.inverse_transform([pred.argmax()])

    return label[0]


input=["i am very happy","i am so worried","i am good","i hate you clearly","i am disgusted by you"]
for i in input:
  print(f"Your text : {i}")
  print("Emotion :",predict_emotion(i))

Your text : i am very happy
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Emotion : fear
Your text : i am so worried
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Emotion : fear
Your text : i am good
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Emotion : disgust
Your text : i hate you clearly
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Emotion : fear
Your text : i am disgusted by you
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Emotion : sadness


The model obivously didn't get trained well, and we have to improvise this